# Hugging Face STT Experiment

This notebook tests a small local automatic speech recognition model for teleMER call audio. The current model is `openai/whisper-base`, a multilingual checkpoint with about 74M parameters that is much lighter than large-v3-turbo.

Use synthetic or consented audio only. Do not place production customer recordings in this experiment.

## 1. Install and Import Dependencies

This installs the local inference stack. Model weights are downloaded from Hugging Face the first time the model is loaded.

In [18]:
%pip install -q transformers accelerate torch soundfile scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import time
from pathlib import Path

import soundfile as sf
import torch
from scipy.signal import resample_poly
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
AUDIO_PATH = Path("artifacts/tts_sample.wav")
TARGET_SAMPLE_RATE = 16_000

print(f"Using device: {DEVICE}")
print(f"Audio path: {AUDIO_PATH.resolve()}")

Using device: cpu
Audio path: C:\dev\telemer\experiments\artifacts\tts_sample.wav


## 2. Select and Load the STT Model

Use Whisper small as the Hindi baseline. It is larger than Whisper base but still much lighter than large-v3-turbo, and should provide better Hindi recognition and script selection. Hindi is forced during decoding.

In [25]:
MODEL_ID = "openai/whisper-small"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(DEVICE)
processor = AutoProcessor.from_pretrained(MODEL_ID)

transcriber = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=TORCH_DTYPE,
    device=DEVICE,
)

print(f"Loaded {MODEL_ID}")

c:\dev\telemer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--openai--whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 479/479 [00:03<00:00, 133.71it/s]


Loaded openai/whisper-small


## 3. Inspect the Input Audio

The speech model expects a readable audio file. Confirm the sample rate, channel count, and duration before transcription.

In [20]:
if not AUDIO_PATH.exists():
    raise FileNotFoundError(
        f"Audio file not found: {AUDIO_PATH}. Generate it with tts_engine.py or set AUDIO_PATH."
    )

audio_info = sf.info(AUDIO_PATH)
audio_array, source_sample_rate = sf.read(
    AUDIO_PATH,
    dtype="float32",
    always_2d=False,
)
if audio_array.ndim == 2:
    audio_array = audio_array.mean(axis=1)

if source_sample_rate != TARGET_SAMPLE_RATE:
    audio_array = resample_poly(
        audio_array,
        TARGET_SAMPLE_RATE,
        source_sample_rate,
    ).astype("float32")

def make_audio_input() -> dict:
    return {
        "raw": audio_array,
        "sampling_rate": TARGET_SAMPLE_RATE,
    }

print({
    "source_sample_rate_hz": source_sample_rate,
    "model_sample_rate_hz": TARGET_SAMPLE_RATE,
    "channels": audio_info.channels,
    "source_frames": audio_info.frames,
    "duration_seconds": round(len(audio_array) / TARGET_SAMPLE_RATE, 2),
})

{'source_sample_rate_hz': 24000, 'model_sample_rate_hz': 16000, 'channels': 1, 'source_frames': 231000, 'duration_seconds': 9.62}


## 4. Transcribe the Audio

Force Hindi transcription for this audio. Explicit language selection prevents Whisper from auto-detecting a closely related language such as Urdu.

In [26]:
start_time = time.perf_counter()
result = transcriber(
    make_audio_input(),
    generate_kwargs={
        "language": "hindi",
        "task": "transcribe",
    },
)
inference_seconds = time.perf_counter() - start_time

transcript = result["text"].strip()
print(f"Transcript: {transcript}")
print(f"Inference time: {inference_seconds:.2f} seconds")

Transcript: हिंदी पार्ठ्यपुस्तके विभिन्न शिक्षा भोरों और कक्षाओं के अनुसार उपलब्द है, प्रमुक् स्रोथ और जानकारी नीचे दीगाई है.
Inference time: 123.55 seconds


## 5. Get Timestamps

Timestamps are useful for debugging interruptions, silence detection, and mapping extracted answers back to the customer's response.

In [ ]:
timestamped_result = transcriber(
    make_audio_input(),
    return_timestamps=True,
    generate_kwargs={
        "language": "hindi",
        "task": "transcribe",
    },
)

for chunk in timestamped_result.get("chunks", []):
    start, end = chunk["timestamp"]
    print(f"{start:.2f}s - {end:.2f}s: {chunk['text'].strip()}")

### Optional Apple Silicon path

`mlx-audio` and `mlx-community/Qwen3-ASR-0.6B-8bit` are intended for Apple Silicon. They are not used by this Windows notebook. Keep the Transformers path above for this machine.

## 6. Candidate Comparison

Use the same Indian-English test set to choose a production model. Compare word error rate (WER), real-time factor, memory use, and behavior on insurance names, policy numbers, dates, and yes/no answers.

- `openai/whisper-base`: current small multilingual baseline for this Windows machine.
- `openai/whisper-small`: next accuracy step if base is not accurate enough; it uses more memory.
- `openai/whisper-large-v3-turbo`: stronger accuracy but too large for the current machine.
- `distil-whisper/distil-large-v3`: fast English-only option, but still much larger than Whisper base.
- `nvidia/parakeet-tdt-0.6b-v2`: fast English ASR, but uses NeMo and is more GPU/Linux-oriented.

Do not choose from benchmark scores alone. Test consented recordings from the target phone channel and accents.